In [2]:
import pandas as pd
import shutil
import os
import numpy as np
import matplotlib.pyplot as plt
import onekey_algo.custom.components as okcomp
from onekey_algo import get_param_in_cwd

plt.rcParams['figure.dpi'] = 300
sel_m = get_param_in_cwd('sel_model')
model_names = get_param_in_cwd('summary_models')
# 获取配置
task = get_param_in_cwd('task_column') or 'label'
bst_model = get_param_in_cwd('sel_model') or 'LR'
labelf = get_param_in_cwd('label_file')
group_info = get_param_in_cwd('dataset_column') or 'group'
figsize = (8, 6)
# 读取label文件。
labels = [task]
label_data_ = pd.read_csv(labelf)
label_data_['ID'] = label_data_['ID'].map(lambda x: os.path.splitext(x)[0])
label_data_ = label_data_[['ID', group_info, task]]
label_data_ = label_data_.dropna(axis=0)
all_res = []
ids = label_data_['ID']
print(label_data_.columns)
label_data = label_data_[['ID'] + labels]
label_data

Index(['ID', 'group', 'label'], dtype='object')


,ID,label
0,陈博201803-202302,2.140
1,齐明旭201609-201802,1.386
2,姜心忠202202-202403,1.705
3,司振哲201906-202302,0.000
4,胡轩铖202208-202407,0.000
...,...,...
300,张颖泰20230525-20241024,1.705
301,赵恩笛20230915-20250105,1.099
302,赵鹏20200521-20241107,0.693
303,赵志男20230401-20241112,0.000


In [6]:
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, mean_absolute_error


def get_period(x):
    if x < 24:
        return '< 2 years'
    elif x < 48:
        return '2-4 years'
    else:
        return '>= 4 years'
label_data = pd.read_csv('data/clinical.csv')[['ID', 'Elapsed_time', 'label']]
label_data['Elapsed_time'] = label_data['Elapsed_time'].map(lambda x: get_period(x))

for subset in get_param_in_cwd('subsets'):
    ALL_results = []
    for mn in  model_names:
        r = pd.read_csv(f"./results/{mn}_{sel_model}_{subset}.csv")
#         display(r)
        r.columns = ['ID', 'Prediction']
        r['Signature'] = mn
        r['Count'] = 1
        ALL_results.append(r)
    ALL_results = pd.concat(ALL_results, axis=0)
    ALL_results =pd.merge(ALL_results, label_data, on='ID', how='inner')
    ALL_results['MSE'] = (ALL_results['label'] - ALL_results['Prediction']) ** 2
    ALL_results['MAPE'] = np.abs(ALL_results['label'] - ALL_results['Prediction'])
#     display(ALL_results)
    results = ALL_results.groupby(['Signature', 'Elapsed_time']).mean().drop('Prediction', axis=1)
    results['Count'] = ALL_results.groupby(['Signature', 'Elapsed_time']).sum()['Count']
    results.to_csv(f'results/subgroup_{subset}.csv', index=True)
    display(results)

FileNotFoundError: [Errno 2] No such file or directory: './results/Clinical_vgg19_train.csv'

In [4]:
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, mean_absolute_error


def get_period(x):
    if x < 24:
        return '< 2 years'
    elif x < 48:
        return '2-4 years'
    else:
        return '>= 4 years'
label_data = pd.read_csv('data/clinical.csv')[['ID', 'Elapsed_time', 'label']]
label_data['Elapsed_time'] = label_data['Elapsed_time'].map(lambda x: get_period(x))

for sel_model in ['SVM', 'RandomForest', 'resnet50', 'densenet121', 'vgg19']:
    for subset in get_param_in_cwd('subsets'):
        for mn in  model_names:
            ALL_results = []
            try:
                r = pd.read_csv(f"./results/{mn}_{sel_m[mn]}_{subset}.csv")
        #         display(r)
                r.columns = ['ID', 'Prediction']
                r['Signature'] = mn
                r['Count'] = 1
                ALL_results.append(r)
                ALL_results = pd.concat(ALL_results, axis=0)
                ALL_results =pd.merge(ALL_results, label_data, on='ID', how='inner')
                ALL_results['MSE'] = (ALL_results['label'] - ALL_results['Prediction']) ** 2
                ALL_results['MAPE'] = np.abs(ALL_results['label'] - ALL_results['Prediction'])
            #     display(ALL_results)
                results = ALL_results.groupby(['Signature', 'Elapsed_time']).mean().drop('Prediction', axis=1)
                results['Count'] = ALL_results.groupby(['Signature', 'Elapsed_time']).sum()['Count']
                results.to_csv(f'results/subgroup_{mn}_{sel_model}_{subset}.csv', index=True)
                display(results)
            except:
                pass

Count  label    MSE   MAPE
Signature Elapsed_time                            
Clinical  2-4 years        54  0.863  0.310  0.376
          < 2 years        51  0.395  0.200  0.308
          >= 4 years       63  1.342  0.147  0.284

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        54  0.863  0.726  0.776
          < 2 years        51  0.395  0.723  0.765
          >= 4 years       63  1.342  1.251  1.000

Count  label    MSE   MAPE
Signature Elapsed_time                            
DLElapsed 2-4 years        54  0.863  0.317  0.496
          < 2 years        51  0.395  0.228  0.412
          >= 4 years       63  1.342  0.408  0.509

Count  label    MSE   MAPE
Signature Elapsed_time                            
Combined  2-4 years        54  0.863  0.143  0.250
          < 2 years        51  0.395  0.107  0.236
          >= 4 years       63  1.342  0.179  0.313

Count  label    MSE   MAPE
Signature Elapsed_time                            
Clinical  2-4 years        22  1.060  0.451  0.510
          < 2 years        28  0.543  0.250  0.379
          >= 4 years       21  1.491  0.743  0.625

...

Count  label    MSE   MAPE
Signature Elapsed_time                            
Combined  2-4 years        22  1.060  0.307  0.362
          < 2 years        28  0.543  0.188  0.339
          >= 4 years       21  1.491  0.817  0.650

Count  label    MSE   MAPE
Signature Elapsed_time                            
Clinical  2-4 years        14  0.685  0.623  0.605
          < 2 years        33  0.467  0.605  0.609
          >= 4 years        2  1.733  0.641  0.737

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        14  0.685  0.505  0.639
          < 2 years        33  0.467  0.582  0.704
          >= 4 years        2  1.733  1.993  1.173

Count  label    MSE   MAPE
Signature Elapsed_time                            
DLElapsed 2-4 years        14  0.685  0.487  0.632
          < 2 years        33  0.467  0.408  0.547
          >= 4 years        2  1.733  2.079  1.082

Count  label    MSE   MAPE
Signature Elapsed_time                            
Combined  2-4 years        14  0.685  0.410  0.451
          < 2 years        33  0.467  0.212  0.333
          >= 4 years        2  1.733  0.808  0.896